<a href="https://colab.research.google.com/github/SlightlyOffset/AI-companion/blob/main/colab_bridge/XTTS_Bridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ AI Companion: XTTS v2 Remote Voice Bridge

### 🛠️ Setup Instructions
1. **GPU**: `Runtime` > `Change runtime type` > **T4 GPU**.
2. **Ngrok**: Add `NGROK_TOKEN` to **Secrets** (key icon) and enable "Notebook access".
3. **Run All**: `Ctrl + F9`.

**Note**: This version uses PyTorch 2.5.1 to ensure compatibility with XTTS v2.

In [1]:
# @title 1. Prepare Environment
import os

print("Installing Python 3.10...")
!sudo apt-get update -qq
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev > /dev/null

print("Creating virtual environment...")
if not os.path.exists("xtts_env"):
    !python3.10 -m venv xtts_env

print("Installing XTTS dependencies (3-5 minutes)...")
!./xtts_env/bin/pip install -q -U pip
# Downgrade torch to 2.5.1 to avoid the PyTorch 2.6 weights_only security error
print("Installing stable PyTorch 2.5.1...")
!./xtts_env/bin/pip install -q fastapi uvicorn torch==2.5.1 torchaudio==2.5.1 transformers==4.33.0 TTS python-multipart

print("\n✅ Environment Ready!")

Installing Python 3.10...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Creating virtual environment...
Installing XTTS dependencies (3-5 minutes)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.0 MB/s eta 0:00:00
Installing stable PyTorch 2.5.1...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Install

In [2]:
# @title 2. Create Bridge Script
with open("bridge_server.py", "w") as f:
    f.write("""
import torch
import os
import shutil
import uuid
from typing import List
from TTS.api import TTS
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import FileResponse
import uvicorn

print("-> Loading XTTS v2 model... (This may take a minute)")
device = "cuda" if torch.cuda.is_available() else "cpu"
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Model Loaded.")

app = FastAPI()

@app.post("/generate_tts")
async def generate_tts_endpoint(
    text: str = Form(...),
    language: str = Form("en"),
    speaker_files: List[UploadFile] = File(...)
):
    job_id = str(uuid.uuid4())
    temp_voice_paths = []
    output_path = f"out_{job_id}.wav"

    for i, upload in enumerate(speaker_files):
        temp_path = f"voice_{job_id}_{i}.wav"
        with open(temp_path, "wb") as buffer:
            shutil.copyfileobj(upload.file, buffer)
        temp_voice_paths.append(temp_path)

    try:
        tts_model.tts_to_file(
            text=text,
            speaker_wav=temp_voice_paths,
            language=language,
            file_path=output_path
        )
        return FileResponse(output_path, media_type="audio/wav", filename="speech.wav")
    finally:
        for p in temp_voice_paths:
            if os.path.exists(p): os.remove(p)
""")
print("✅ Bridge script created.")

✅ Bridge script created.


In [3]:
# @title 3. Start XTTS Bridge
import sys
import subprocess
import time

try:
    from pyngrok import ngrok
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyngrok"])
    from pyngrok import ngrok

from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
    ngrok.set_auth_token(NGROK_TOKEN)
except:
    print("❌ ERROR: NGROK_TOKEN not found in Secrets!")

ngrok.kill()
public_url = ngrok.connect(8001).public_url

print("="*50)
print(f"\n🚀 XTTS BRIDGE ONLINE!\n")
print(f"URL: {public_url}\n")
print("="*50)

# Run server
!./xtts_env/bin/uvicorn bridge_server:app --host 0.0.0.0 --port 8001 --log-level error


🚀 XTTS BRIDGE ONLINE!

URL: https://aerobically-meddlesome-ria.ngrok-free.dev

-> Loading XTTS v2 model... (This may take a minute)
 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y
 > Downloading model to /root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2
 99% 1.86G/1.87G [00:29<00:00, 88.7MiB/s]
100% 1.87G/1.87G [00:29<00:00, 63.2MiB/s]
4.37kiB [00:00, 15.1kiB/s]

361kiB [00:00, 1.11MiB/s]
100% 32.0/32.0 [00:00<00:00, 71.6iB/s]
 70% 5.45M/7.75M [00:00<00:00, 54.5MiB/s] > Model's license - CPML
 > Check https://coqui.ai/cpml.txt for more info.
/content/xtts_env/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_no